Facebook data downloaded from Humanitarian Data Exchange on 17th January 2025.
Links for 2020 and 2021-2022 data are: https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/55a51014-0d27-49ae-bf92-c82a570c2c6c/download/movement-range-data-2022-05-22.zip
and https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/3d77ce5c-ab6d-4864-b8a2-c8bafffac4f3/download/movement-range-data-2020-03-01-2020-12-31.zip
which are accessible from:
https://data.humdata.org/dataset/movement-range-maps?

In [ ]:
import pandas as pd
import pycountry
import re
from emu_renewal.inputs import RAW_MOB_PATH, DATA_PATH, get_country_mobility

In [ ]:
data20 = pd.read_csv(RAW_MOB_PATH / "movement-range-data-2020-03-01--2020-12-31.txt", sep="\t", index_col="ds", low_memory=False)
data21_22 = pd.read_csv(RAW_MOB_PATH / "movement-range-2022-05-22.txt", sep="\t", index_col="ds", low_memory=False)
fb_data = pd.concat([data20, data21_22])
fb_data.index = pd.to_datetime(fb_data.index)

In [ ]:
euro_pop_data = pd.read_csv(DATA_PATH / "population/estat_demo_r_d2jan_filtered_en.csv", index_col="geo")
euro_pops = euro_pop_data.loc[euro_pop_data["TIME_PERIOD"] == 2020, "OBS_VALUE"]

In [ ]:
country = "France"
iso2_code = pycountry.countries.get(name=country).alpha_2
iso3_code = pycountry.countries.get(name=country).alpha_3

In [ ]:
pattern = re.compile(f"^{iso2_code}[A-Z]:.+$")

In [ ]:
# Extract the region name
country_pop_map = euro_pops[[i for i in euro_pops.index if pattern.match(i)]]
country_pop_map.index = [i.split(":")[1] for i in country_pop_map.index]

In [ ]:
country_mobility = fb_data[fb_data["country"] == iso3_code]

In [ ]:
# Get rid of non-standard characters from data and mapper
country_pop_map.index = country_pop_map.index.str.replace("Î", "-").str.replace("ô", "-").str.replace("é", "-")
country_mobility.loc[:, "polygon_name"] = country_mobility["polygon_name"].str.replace("Î", "-").str.replace("ô", "-").str.replace("é", "-")

In [ ]:
country_mobility.loc[:, "pop"] = country_mobility["polygon_name"].map(country_pop_map)